# Notebook 2 — Basic GANs

**Phase 2** deliverable. Learning objectives:
- Vanilla GAN architecture
- DCGAN with best practices
- Conditional GAN for controlled generation
- Detecting & addressing mode collapse
- Wasserstein GAN for training stability

Complete the TODOs in `models/gans/` and `training/losses.py` (adversarial,
WGAN, and gradient-penalty losses) before running the training cells. Run
`pytest -m gan -v` to check your progress.


In [1]:
import sys
from pathlib import Path

# Make `generative_art_studio` importable without `pip install -e .`
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
# Ensure the project root is used when the notebook is launched from the notebooks folder.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from generative_art_studio.utils import set_seed, plot_image_grid
from generative_art_studio.config import DEVICE

set_seed(42)
print(f"Using device: {DEVICE}")


Using device: cpu


In [2]:
from generative_art_studio.data import SyntheticImageDataset, get_dataloader

dataset = SyntheticImageDataset(num_samples=256, image_size=64, num_classes=2)
dataloader = get_dataloader(dataset, batch_size=32)


## 1. Vanilla GAN (fully-connected) — reference architecture

In [3]:
from generative_art_studio.models.gans import VanillaGenerator, VanillaDiscriminator
from generative_art_studio.training.train_gan import train_gan
from torch.optim import Adam

latent_dim = 64
generator = VanillaGenerator(latent_dim=latent_dim).to(DEVICE)
discriminator = VanillaDiscriminator().to(DEVICE)

g_opt = Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

history = train_gan(generator, discriminator, dataloader, g_opt, d_opt, latent_dim, device=DEVICE, epochs=3)


GAN epoch 1/3:   0%|          | 0/8 [00:00<?, ?it/s]

GAN epoch 2/3:   0%|          | 0/8 [00:00<?, ?it/s]

GAN epoch 3/3:   0%|          | 0/8 [00:00<?, ?it/s]

In [4]:
import matplotlib.pyplot as plt
plt.plot(history.g_loss, label="G loss")
plt.plot(history.d_loss, label="D loss")
plt.legend(); plt.title("Vanilla GAN training — watch for D loss collapsing near 0 (mode collapse warning sign)")
plt.show()

from generative_art_studio.utils.latent_space import sample_latent
z = sample_latent(16, latent_dim, DEVICE)
samples = generator(z)
plot_image_grid(samples.detach().cpu(), nrow=4, title="Vanilla GAN samples")


C:\Users\jbisw\AppData\Local\Temp\ipykernel_29552\1744527637.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 400x400 with 1 Axes>

## 2. DCGAN

Implement `DCGANGenerator`/`DCGANDiscriminator` in `models/gans/dcgan.py`,
then repeat the training loop above with these models (remember to call
`.apply(weights_init_dcgan)` right after construction) and `latent_dim=100`.


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from generative_art_studio.models.gans import DCGANGenerator, DCGANDiscriminator, weights_init_dcgan

dc_latent_dim = 100
dc_generator = DCGANGenerator(latent_dim=dc_latent_dim, feature_maps=32).to(DEVICE)
dc_discriminator = DCGANDiscriminator(feature_maps=32).to(DEVICE)
dc_generator.apply(weights_init_dcgan)
dc_discriminator.apply(weights_init_dcgan)

# TODO: train dc_generator/dc_discriminator the same way as the vanilla GAN above,
# then visualize samples and compare training-curve stability.

import torch.optim as optim
import matplotlib.pyplot as plt

criterion = nn.BCELoss()
lr = 2e-4
beta1 = 0.5

optimizerD = optim.Adam(dc_discriminator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(dc_generator.parameters(), lr=lr, betas=(beta1, 0.999))

fixed_noise = torch.randn(64, dc_latent_dim, device=DEVICE)
real_label, fake_label = 1.0, 0.0

num_epochs = 25
dc_G_losses, dc_D_losses = [], []

for epoch in range(num_epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(DEVICE)
        b_size = real_imgs.size(0)

        # ---- Train Discriminator: maximize log(D(x)) + log(1 - D(G(z))) ----
        dc_discriminator.zero_grad()
        label = torch.full((b_size, 1), real_label, device=DEVICE)
        output = dc_discriminator(real_imgs)
        lossD_real = criterion(output, label)
        lossD_real.backward()

        noise = torch.randn(b_size, dc_latent_dim, device=DEVICE)
        fake = dc_generator(noise)
        label.fill_(fake_label)
        output = dc_discriminator(fake.detach())
        lossD_fake = criterion(output, label)
        lossD_fake.backward()

        lossD = lossD_real + lossD_fake
        optimizerD.step()

        # ---- Train Generator: maximize log(D(G(z))) ----
        dc_generator.zero_grad()
        label.fill_(real_label)  # generator wants D to think these are real
        output = dc_discriminator(fake)
        lossG = criterion(output, label)
        lossG.backward()
        optimizerG.step()

        dc_G_losses.append(lossG.item())
        dc_D_losses.append(lossD.item())

    print(f"Epoch [{epoch+1}/{num_epochs}]  Loss_D: {lossD.item():.4f}  Loss_G: {lossG.item():.4f}")

# ---- Visualize samples ----
dc_generator.eval()
with torch.no_grad():
    samples = dc_generator(fixed_noise).cpu()
dc_generator.train()

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for ax, img in zip(axes.flatten(), samples):
    img = (img * 0.5 + 0.5).permute(1, 2, 0).numpy()  # unnormalize from [-1,1]
    ax.imshow(img.squeeze(), cmap="gray" if img.shape[-1] == 1 else None)
    ax.axis("off")
plt.tight_layout()
plt.show()

# ---- Compare training-curve stability ----
plt.figure(figsize=(8, 4))
plt.plot(dc_G_losses, label="DCGAN Generator")
plt.plot(dc_D_losses, label="DCGAN Discriminator")
# if you saved vanilla-GAN losses earlier as G_losses/D_losses, overlay them:
# plt.plot(G_losses, label="Vanilla GAN Generator", linestyle="--")
# plt.plot(D_losses, label="Vanilla GAN Discriminator", linestyle="--")
plt.title("Training Loss Comparison")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.show()



Epoch [1/25]  Loss_D: 0.3370  Loss_G: 4.7267
Epoch [2/25]  Loss_D: 0.1571  Loss_G: 5.1629
Epoch [3/25]  Loss_D: 0.0985  Loss_G: 6.2641
Epoch [4/25]  Loss_D: 0.0649  Loss_G: 6.7907
Epoch [5/25]  Loss_D: 0.0459  Loss_G: 6.7114
Epoch [6/25]  Loss_D: 0.0414  Loss_G: 7.6561
Epoch [7/25]  Loss_D: 0.0327  Loss_G: 7.2868
Epoch [8/25]  Loss_D: 0.0234  Loss_G: 7.8006
Epoch [9/25]  Loss_D: 0.0256  Loss_G: 8.0905
Epoch [10/25]  Loss_D: 0.0141  Loss_G: 6.4067
Epoch [11/25]  Loss_D: 0.0291  Loss_G: 12.7654
Epoch [12/25]  Loss_D: 0.0171  Loss_G: 7.4669
Epoch [13/25]  Loss_D: 0.0189  Loss_G: 10.1222
Epoch [14/25]  Loss_D: 0.0205  Loss_G: 9.0020
Epoch [15/25]  Loss_D: 0.0155  Loss_G: 8.5227
Epoch [16/25]  Loss_D: 0.0201  Loss_G: 9.9712
Epoch [17/25]  Loss_D: 0.0153  Loss_G: 11.2546
Epoch [18/25]  Loss_D: 0.0223  Loss_G: 16.7917
Epoch [19/25]  Loss_D: 0.0028  Loss_G: 9.7598
Epoch [20/25]  Loss_D: 0.0384  Loss_G: 14.1780
Epoch [21/25]  Loss_D: 0.0126  Loss_G: 7.1698
Epoch [22/25]  Loss_D: 0.1127  Loss_G:

C:\Users\jbisw\AppData\Local\Temp\ipykernel_29552\47502959.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\jbisw\AppData\Local\Temp\ipykernel_29552\47502959.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Conditional GAN

Implement `ConditionalGenerator`/`ConditionalDiscriminator` in
`models/gans/conditional_gan.py`, then generate samples per class label to
demonstrate controlled generation.


In [7]:
from generative_art_studio.models.gans import ConditionalGenerator, ConditionalDiscriminator

cgan_gen = ConditionalGenerator(latent_dim=64, num_classes=2).to(DEVICE)

# TODO: train, then compare generator output for label=0 vs label=1 with the same z
# z = sample_latent(8, 64, DEVICE)
# samples_label0 = cgan_gen(z, torch.zeros(8, dtype=torch.long, device=DEVICE))
# samples_label1 = cgan_gen(z, torch.ones(8, dtype=torch.long, device=DEVICE))

cgan_disc = ConditionalDiscriminator(num_classes=2).to(DEVICE)

training_cgan = True  # Set to False if you want to skip training and just visualize samples

if training_cgan:
    criterion = nn.BCELoss()
    optimizerD = optim.Adam(cgan_disc.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optimizerG = optim.Adam(cgan_gen.parameters(), lr=2e-4, betas=(0.5, 0.999))

    num_epochs = 25
    for epoch in range(num_epochs):
        for real_imgs, labels in dataloader:  # dataloader must yield (image, class_label)
            real_imgs, labels = real_imgs.to(DEVICE), labels.to(DEVICE)
            b_size = real_imgs.size(0)

            # ---- Train Discriminator ----
            cgan_disc.zero_grad()
            real_target = torch.full((b_size, 1), 1.0, device=DEVICE)
            output_real = cgan_disc(real_imgs, labels)
            lossD_real = criterion(output_real, real_target)

            z = sample_latent(b_size, 64, DEVICE)
            fake_labels = torch.randint(0, 2, (b_size,), device=DEVICE)
            fake_imgs = cgan_gen(z, fake_labels)
            fake_target = torch.full((b_size, 1), 0.0, device=DEVICE)
            output_fake = cgan_disc(fake_imgs.detach(), fake_labels)
            lossD_fake = criterion(output_fake, fake_target)

            lossD = lossD_real + lossD_fake
            lossD.backward()
            optimizerD.step()

            # ---- Train Generator ----
            cgan_gen.zero_grad()
            output = cgan_disc(fake_imgs, fake_labels)
            lossG = criterion(output, real_target)  # wants discriminator to say "real"
            lossG.backward()
            optimizerG.step()

        print(f"Epoch [{epoch+1}/{num_epochs}]  Loss_D: {lossD.item():.4f}  Loss_G: {lossG.item():.4f}")

# ---- Compare label=0 vs label=1 output for the same z ----
cgan_gen.eval()
with torch.no_grad():
    z = sample_latent(8, 64, DEVICE)
    samples_label0 = cgan_gen(z, torch.zeros(8, dtype=torch.long, device=DEVICE))
    samples_label1 = cgan_gen(z, torch.ones(8, dtype=torch.long, device=DEVICE))
cgan_gen.train()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    img0 = (samples_label0[i] * 0.5 + 0.5).cpu().permute(1, 2, 0).numpy()
    img1 = (samples_label1[i] * 0.5 + 0.5).cpu().permute(1, 2, 0).numpy()
    axes[0, i].imshow(img0.squeeze(), cmap="gray" if img0.shape[-1] == 1 else None)
    axes[0, i].axis("off")
    axes[1, i].imshow(img1.squeeze(), cmap="gray" if img1.shape[-1] == 1 else None)
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("label=0")
axes[1, 0].set_ylabel("label=1")
plt.tight_layout()
plt.show()




Epoch [1/25]  Loss_D: 1.2572  Loss_G: 0.7576
Epoch [2/25]  Loss_D: 1.7524  Loss_G: 1.1676
Epoch [3/25]  Loss_D: 1.4754  Loss_G: 1.0559
Epoch [4/25]  Loss_D: 1.0675  Loss_G: 1.2448
Epoch [5/25]  Loss_D: 1.1104  Loss_G: 1.7487
Epoch [6/25]  Loss_D: 1.2847  Loss_G: 1.3312
Epoch [7/25]  Loss_D: 1.3950  Loss_G: 0.8353
Epoch [8/25]  Loss_D: 1.2421  Loss_G: 1.0737
Epoch [9/25]  Loss_D: 1.3083  Loss_G: 0.9067
Epoch [10/25]  Loss_D: 1.5059  Loss_G: 0.7329
Epoch [11/25]  Loss_D: 1.3183  Loss_G: 1.4397
Epoch [12/25]  Loss_D: 1.1646  Loss_G: 2.9627
Epoch [13/25]  Loss_D: 1.0246  Loss_G: 3.3495
Epoch [14/25]  Loss_D: 0.7246  Loss_G: 3.2169
Epoch [15/25]  Loss_D: 0.9372  Loss_G: 3.8108
Epoch [16/25]  Loss_D: 0.6411  Loss_G: 4.2282
Epoch [17/25]  Loss_D: 0.4073  Loss_G: 4.3911
Epoch [18/25]  Loss_D: 0.5812  Loss_G: 5.2035
Epoch [19/25]  Loss_D: 0.5119  Loss_G: 16.6983
Epoch [20/25]  Loss_D: 0.5439  Loss_G: 6.9122
Epoch [21/25]  Loss_D: 0.5127  Loss_G: 5.3763
Epoch [22/25]  Loss_D: 0.7219  Loss_G: 8.4

C:\Users\jbisw\AppData\Local\Temp\ipykernel_29552\1170050261.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Wasserstein GAN (WGAN-GP)

Implement `WGANCritic` (`models/gans/wgan.py`) and `wgan_critic_loss` /
`wgan_generator_loss` / `gradient_penalty` (`training/losses.py`), then
write your own critic/generator training loop below — unlike the BCE GAN,
WGAN trains the critic `N_CRITIC` steps per generator step (see
`config.N_CRITIC`).


In [ ]:
from generative_art_studio.models.gans import WGANCritic
from generative_art_studio.training.losses import wgan_critic_loss, wgan_generator_loss, gradient_penalty
from generative_art_studio import config

wgan_gen = DCGANGenerator(latent_dim=100, feature_maps=32).to(DEVICE)  # generator arch is unchanged
critic = WGANCritic(feature_maps=32).to(DEVICE)

# TODO: implement the WGAN-GP training loop:
#   for N_CRITIC steps: update critic using wgan_critic_loss + WGAN_GP_LAMBDA * gradient_penalty
#   then: one generator update using wgan_generator_loss
# Compare training-curve smoothness against the vanilla/DCGAN BCE-based runs above.


## Reflection (for your Technical Report)

- Did you observe **mode collapse** in any model (e.g. all generated
  samples looking nearly identical)? Show a sample grid as evidence.
- Compare G/D loss curve *shape* across vanilla GAN, DCGAN, and WGAN — which
  was most stable, and why (tie back to what each loss function optimizes)?
- What did conditioning on class label change about controllability?
